# Day 12：vLLM 推理加速原理

前面学的都是 Agent 的应用层——怎么用 LLM。今天换个角度，学 LLM 推理的底层原理。

为什么要学这个？
1. **面试加分**：能讲出 PagedAttention 原理的候选人非常少
2. **工程直觉**：理解瓶颈在哪，才能做出正确的优化决策
3. **简历差异化**：大多数人只会调 API，你懂底层

## 一、KV Cache 是什么

Transformer 生成文本时，每生成一个新 Token，都需要和**所有历史 Token** 做 Attention。

```
生成第 5 个 Token 时：
  Q5 需要和 K1,K2,K3,K4 做 Attention
  Q5 需要和 V1,V2,V3,V4 做加权求和

生成第 6 个 Token 时：
  Q6 需要和 K1,K2,K3,K4,K5 做 Attention
```

如果每次都重新计算 K1-K5，太浪费了！所以把之前算过的 K 和 V 缓存起来——这就是 **KV Cache**。

```
没有 KV Cache：每步重新计算所有 K,V → 每步 O(n)，n 步总计 O(n²)
有 KV Cache：只计算新 Token 的 K,V，查表获取历史 → 每步 O(1)，n 步总计 O(n)
```

In [ ]:
# ===== KV Cache 空间计算 =====
# 这是面试高频考点：给你模型参数，算 KV Cache 占多少显存

def calc_kv_cache_size(
    num_layers: int,
    num_kv_heads: int,
    head_dim: int,
    seq_len: int,
    dtype_bytes: float = 2,  # FP16=2, FP8=1, INT4=0.5
    batch_size: int = 1,
) -> float:
    """
    KV Cache 显存计算公式
    
    公式：KV Cache = 2 × num_layers × num_kv_heads × head_dim × seq_len × dtype_bytes × batch_size
    
    其中 2 是因为有 K 和 V 两份
    """
    bytes_total = 2 * num_layers * num_kv_heads * head_dim * seq_len * dtype_bytes * batch_size
    gb = bytes_total / (1024 ** 3)
    return gb

# ===== 常见模型的 KV Cache 计算 =====

models = {
    "Qwen2.5-1.5B": {
        "num_layers": 28,
        "num_kv_heads": 2,   # GQA: KV heads 比 Q heads 少
        "head_dim": 64,
    },
    "Qwen2.5-7B": {
        "num_layers": 28,
        "num_kv_heads": 2,
        "head_dim": 128,
    },
    "Qwen2.5-14B": {
        "num_layers": 48,
        "num_kv_heads": 8,
        "head_dim": 128,
    },
    "LLaMA3-8B": {
        "num_layers": 32,
        "num_kv_heads": 8,
        "head_dim": 128,
    },
}

print("KV Cache 显存计算")
print("=" * 70)
print(f"{'模型':<20} {'seq_len':<10} {'FP16 (GB)':<12} {'FP8 (GB)':<12} {'INT4 (GB)'}")
print("-" * 70)

for name, params in models.items():
    for seq_len in [2048, 4096, 8192]:
        fp16 = calc_kv_cache_size(**params, seq_len=seq_len, dtype_bytes=2)
        fp8 = calc_kv_cache_size(**params, seq_len=seq_len, dtype_bytes=1)
        int4 = calc_kv_cache_size(**params, seq_len=seq_len, dtype_bytes=0.5)
        print(f"{name:<20} {seq_len:<10} {fp16:<12.2f} {fp8:<12.2f} {int4:.2f}")
    print()

print("=" * 70)
print("\n关键洞察：")
print("1. seq_len 翻倍，KV Cache 也翻倍——这是长上下文的代价")
print("2. FP8 量化可以把 KV Cache 减半，几乎不影响质量")
print("3. 7B 模型 4096 长度就需要 1.64GB KV Cache（FP16）")
print("4. 批量推理时，KV Cache 乘以 batch_size——这才是真正的显存杀手")

## 二、为什么推理是 Memory-bound

LLM 推理分为两个阶段：

### Prefill 阶段（处理输入）
```
输入: [Token1, Token2, ..., Token_n]
→ 并行计算所有 Token 的 K,V
→ 计算密集型（Compute-bound）
→ GPU 算力利用率高
```

### Decode 阶段（生成输出）
```
每步只生成 1 个 Token
→ 读取整个 KV Cache（可能几 GB）
→ 只做一次矩阵-向量乘法
→ 计算量很小，但显存带宽是瓶颈
→ Memory-bound，GPU 算力利用率只有 5-10%
```

这就是为什么 LLM 推理慢——**不是算力不够，是显存带宽不够**。

类比：你有一个超级快的 CPU，但硬盘读取速度很慢。
CPU 大部分时间在等数据从硬盘加载到内存。

In [ ]:
# ===== 推理瓶颈分析 =====

def analyze_bottleneck(
    model_params_b: float,    # 模型参数量（B = 十亿）
    seq_len: int,
    gpu_tflops: float,        # GPU 算力（TFLOPS）
    gpu_bandwidth_gbs: float, # GPU 显存带宽（GB/s）
):
    """
    分析推理的瓶颈在哪
    
    Decode 阶段每步：
    - 计算量：~2 * params 次浮点运算（矩阵-向量乘法）
    - 数据量：~2 * params * dtype_bytes 字节（读取权重）
    """
    # 计算量（FLOPS）
    compute_flops = 2 * model_params_b * 1e9  # 简化为 2*params
    compute_time_ms = compute_flops / (gpu_tflops * 1e12) * 1000
    
    # 数据量（Bytes）- 读取模型权重
    data_bytes = 2 * model_params_b * 1e9  # FP16: 2 bytes per param
    memory_time_ms = data_bytes / (gpu_bandwidth_gbs * 1e9) * 1000
    
    # 瓶颈判断
    if compute_time_ms > memory_time_ms:
        bottleneck = "Compute-bound"
        utilization = 100
    else:
        bottleneck = "Memory-bound"
        utilization = compute_time_ms / memory_time_ms * 100
    
    return {
        "compute_time_ms": compute_time_ms,
        "memory_time_ms": memory_time_ms,
        "bottleneck": bottleneck,
        "gpu_utilization": utilization,
    }

# 不同 GPU 的分析
gpus = {
    "RTX 4090": {"tflops": 82.6, "bandwidth_gbs": 1008},
    "A100-80GB": {"tflops": 312, "bandwidth_gbs": 2039},
    "H100": {"tflops": 989, "bandwidth_gbs": 3350},
}

print("推理瓶颈分析（Decode 阶段，batch_size=1）")
print("=" * 70)

for model_name, params_b in [("Qwen2.5-7B", 7), ("Qwen2.5-14B", 14), ("LLaMA3-70B", 70)]:
    print(f"\n{model_name} ({params_b}B 参数):")
    print(f"  {'GPU':<15} {'计算时间':<12} {'显存时间':<12} {'瓶颈':<15} {'GPU利用率'}")
    print(f"  {'-'*60}")
    for gpu_name, gpu_params in gpus.items():
        result = analyze_bottleneck(params_b, 4096, gpu_params["tflops"], gpu_params["bandwidth_gbs"])
        print(f"  {gpu_name:<15} {result['compute_time_ms']:.3f}ms      "
              f"{result['memory_time_ms']:.3f}ms      {result['bottleneck']:<15} "
              f"{result['gpu_utilization']:.1f}%")

print("\n" + "=" * 70)
print("\n结论：")
print("- 小模型（7B）在任何 GPU 上都是 Memory-bound")
print("- 大模型（70B）在高端 GPU 上仍然是 Memory-bound")
print("- GPU 算力利用率通常只有 5-10%")
print("- 提升推理速度的关键：提高显存带宽利用率，而不是增加算力")

## 三、PagedAttention

这是 vLLM 的核心创新，灵感来自操作系统的**虚拟内存分页**机制。

### 传统方式的问题

```
请求1: 最大长度 2048 → 预分配 2048 的连续显存
  实际只用了 512 → 浪费 75%

请求2: 最大长度 2048 → 预分配 2048 的连续显存
  实际只用了 1024 → 浪费 50%
```

预分配导致大量显存碎片和浪费。显存利用率只有 50-60%。

### PagedAttention 的方案

```
把 KV Cache 分成固定大小的「页」（如每页 16 个 Token）

请求1: 需要 512 Token → 分配 32 页（32×16=512）
请求2: 需要 1024 Token → 分配 64 页
请求3: 需要 256 Token → 分配 16 页

页可以不连续存储，通过页表映射
```

类比 OS：
- 传统方式 = 进程需要连续内存 → 内存碎片严重
- PagedAttention = 进程用虚拟内存 → 页表映射 → 无碎片

In [ ]:
# ===== PagedAttention 模拟 =====

class PagedKVCache:
    """
    PagedAttention 的 KV Cache 管理
    
    核心思想：把 KV Cache 分成固定大小的页，按需分配
    """
    
    def __init__(self, page_size=16, total_pages=100):
        self.page_size = page_size    # 每页能存多少 Token 的 KV
        self.total_pages = total_pages
        self.free_pages = list(range(total_pages))  # 空闲页列表
        self.allocations = {}  # request_id → [page_ids]
    
    def allocate(self, request_id: str, num_tokens: int) -> list[int]:
        """为请求分配页"""
        num_pages = (num_tokens + self.page_size - 1) // self.page_size
        
        if len(self.free_pages) < num_pages:
            raise MemoryError(f"显存不足！需要 {num_pages} 页，只剩 {len(self.free_pages)} 页")
        
        # 从空闲列表分配（不需要连续！）
        pages = []
        for _ in range(num_pages):
            pages.append(self.free_pages.pop(0))
        
        self.allocations[request_id] = pages
        return pages
    
    def free(self, request_id: str):
        """释放请求的页"""
        if request_id in self.allocations:
            pages = self.allocations.pop(request_id)
            self.free_pages.extend(pages)
    
    def get_physical_address(self, request_id: str, token_idx: int) -> tuple[int, int]:
        """把逻辑地址（token_idx）转成物理地址（page_id, offset）"""
        pages = self.allocations[request_id]
        page_idx = token_idx // self.page_size
        offset = token_idx % self.page_size
        return pages[page_idx], offset


# 演示
cache = PagedKVCache(page_size=16, total_pages=50)

print("PagedAttention 模拟演示")
print("=" * 50)
print(f"总页数: {cache.total_pages}, 每页: {cache.page_size} Token")
print(f"总容量: {cache.total_pages * cache.page_size} Token")

# 模拟 3 个并发请求
requests = [
    ("req-1", 45),   # 需要 3 页
    ("req-2", 100),  # 需要 7 页
    ("req-3", 30),   # 需要 2 页
]

print("\n分配过程：")
for req_id, tokens in requests:
    pages = cache.allocate(req_id, tokens)
    num_pages = len(pages)
    print(f"  {req_id}: {tokens} Token → {num_pages} 页 {pages}")

print(f"\n已分配: {sum(len(p) for p in cache.allocations.values())} 页")
print(f"空闲: {len(cache.free_pages)} 页")

# 模拟地址转换
print("\n地址转换示例：")
print(f"  req-1 的第 20 个 Token → 物理地址: {cache.get_physical_address('req-1', 20)}")
print(f"  req-2 的第 50 个 Token → 物理地址: {cache.get_physical_address('req-2', 50)}")

# 释放 req-2
print("\n释放 req-2...")
cache.free("req-2")
print(f"空闲: {len(cache.free_pages)} 页")

# 新请求可以复用释放的页
pages = cache.allocate("req-4", 80)
print(f"req-4: 80 Token → {len(pages)} 页 {pages}")

print("\n" + "=" * 50)
print("\nPagedAttention 的优势：")
print("1. 按需分配，没有预分配浪费")
print("2. 页可以不连续，消除碎片")
print("3. 请求结束后页立即可复用")
print("4. 显存利用率从 50-60% 提升到 90%+")

## 四、Continuous Batching

传统 Static Batching 的问题：

```
Batch 中有 3 个请求：
  请求1: 生成 10 个 Token → 第 10 步完成
  请求2: 生成 50 个 Token → 第 50 步完成
  请求3: 生成 5 个 Token  → 第 5 步完成

Static Batching：
  请求3 完成后 GPU 空闲等其他请求
  请求1 完成后 GPU 空闲等请求2
  整体效率很低
```

Continuous Batching（又叫 Iteration-level Scheduling）：

```
每一步结束后：
  完成的请求立即退出，腾出位置
  等待队列中的新请求立即加入
  GPU 始终满载
```

In [ ]:
# ===== Continuous Batching 模拟 =====

def simulate_static_batching(requests):
    """Static Batching：所有请求同时开始、同时结束"""
    max_tokens = max(r["tokens"] for r in requests)
    total_steps = 0
    idle_gpu_steps = 0
    
    print("Static Batching:")
    for step in range(max_tokens):
        active = sum(1 for r in requests if r["tokens"] > step)
        idle = len(requests) - active
        idle_gpu_steps += idle
        total_steps += 1
    
    print(f"  总步数: {max_tokens}")
    print(f"  GPU 空闲步数: {idle_gpu_steps}")
    print(f"  GPU 利用率: {(1 - idle_gpu_steps / (max_tokens * len(requests))) * 100:.1f}%")
    return max_tokens


def simulate_continuous_batching(requests, max_batch_size=3):
    """Continuous Batching：请求完成立即替换"""
    queue = list(requests)
    running = []
    completed = []
    step = 0
    total_gpu_slots = 0
    busy_gpu_slots = 0
    
    print("\nContinuous Batching:")
    
    while queue or running:
        # 填充空位
        while len(running) < max_batch_size and queue:
            req = queue.pop(0)
            req["current"] = 0
            running.append(req)
        
        if not running:
            break
        
        # 执行一步
        step += 1
        total_gpu_slots += max_batch_size
        busy_gpu_slots += len(running)
        
        # 检查完成的请求
        still_running = []
        for req in running:
            req["current"] += 1
            if req["current"] >= req["tokens"]:
                completed.append(req)
            else:
                still_running.append(req)
        running = still_running
    
    print(f"  总步数: {step}")
    print(f"  GPU 利用率: {busy_gpu_slots / total_gpu_slots * 100:.1f}%")
    return step


# 模拟请求
requests = [
    {"id": "req-1", "tokens": 10},
    {"id": "req-2", "tokens": 50},
    {"id": "req-3", "tokens": 5},
]

print("请求列表：")
for r in requests:
    print(f"  {r['id']}: 生成 {r['tokens']} 个 Token")
print()

static_steps = simulate_static_batching(requests)
cont_steps = simulate_continuous_batching(requests)

print("\n" + "=" * 50)
print(f"对比：Static {static_steps} 步 vs Continuous {cont_steps} 步")
print(f"加速比: {static_steps / cont_steps:.2f}x")
print("\nContinuous Batching 的优势：")
print("1. 短请求不用等长请求完成")
print("2. GPU 始终满载，利用率接近 100%")
print("3. 整体吞吐量提升 2-4 倍")

## 五、vLLM 的其他优化

### 5.1 Prefix Caching

如果多个请求有相同的 System Prompt（比如都是"你是一个客服助手"），
vLLM 会缓存这部分的 KV Cache，避免重复计算。

```
请求1: [System Prompt] + [用户问题1]
请求2: [System Prompt] + [用户问题2]
                ↓
System Prompt 的 KV Cache 只算一次，两个请求共享
```

### 5.2 Speculative Decoding（投机解码）

用一个小模型快速生成多个候选 Token，再用大模型验证。

```
小模型（快）: 一次生成 5 个候选 Token
大模型（慢）: 并行验证这 5 个 Token
如果前 3 个都正确 → 一步生成 3 个 Token
```

### 5.3 Tensor Parallelism

把模型切分到多张 GPU 上，并行计算。

```
一张 GPU 放不下 70B 模型
→ 把权重矩阵按列切分到 4 张 GPU
→ 每张 GPU 只算 1/4 的计算量
→ 通过 AllReduce 同步结果
```

In [ ]:
# ===== vLLM vs 原生 HuggingFace 推理对比 =====
# 这是面试中常见的对比问题

comparison = """
┌─────────────────────┬──────────────────────┬──────────────────────┐
│       特性          │   HuggingFace 原生    │      vLLM            │
├─────────────────────┼──────────────────────┼──────────────────────┤
│ KV Cache 管理       │ 连续预分配，碎片严重  │ PagedAttention，无碎片│
│ 显存利用率          │ 50-60%               │ 90%+                 │
│ 批处理策略          │ Static Batching      │ Continuous Batching  │
│ 并发吞吐量          │ 低（GPU 空闲多）      │ 高（GPU 满载）        │
│ Prefix Caching      │ 不支持               │ 支持                 │
│ Tensor Parallel     │ 需手动配置           │ 自动支持              │
│ 量化支持            │ 需要额外配置         │ 原生支持              │
│ OpenAI 兼容 API     │ 不支持               │ 内置支持              │
└─────────────────────┴──────────────────────┴──────────────────────┘
"""

print("vLLM vs HuggingFace 推理对比")
print(comparison)

print("实际性能差异（经验数据）：")
print("  - 单请求延迟：vLLM 略慢 5-10%（有额外调度开销）")
print("  - 并发吞吐：vLLM 快 5-24 倍（取决于并发数）")
print("  - 显存效率：vLLM 能服务更多并发请求")
print("\n结论：")
print("  - 单请求低延迟场景：差异不大")
print("  - 高并发服务场景：vLLM 优势巨大")
print("  - 生产环境部署：几乎必选 vLLM 或类似引擎")

print("\n" + "=" * 50)
print("面试回答模板：")
print("vLLM 通过 PagedAttention 解决了 KV Cache 的显存碎片问题，")
print("借鉴 OS 的虚拟内存分页机制，将 KV Cache 分成固定大小的页，")
print("按需分配、非连续存储，显存利用率从 50-60% 提升到 90%+。")
print("配合 Continuous Batching，高并发吞吐量提升 5-24 倍。")

## 六、动手：用 vLLM 启动本地模型

```bash
# 安装 vLLM
pip install vllm

# 启动 API 服务（OpenAI 兼容）
vllm serve Qwen/Qwen2.5-7B-Instruct \
    --host 0.0.0.0 \
    --port 8000 \
    --max-model-len 4096

# 用 OpenAI SDK 调用
from openai import OpenAI
client = OpenAI(base_url="http://localhost:8000/v1", api_key="none")
response = client.chat.completions.create(
    model="Qwen/Qwen2.5-7B-Instruct",
    messages=[{"role": "user", "content": "你好"}]
)
```

vLLM 内置了 OpenAI 兼容的 API 服务，启动后可以直接用 OpenAI SDK 调用。

## 今日总结

今天从底层理解了 LLM 推理的瓶颈和优化方案。

### 核心知识

| 概念 | 一句话解释 |
|------|----------|
| KV Cache | 缓存历史 Token 的 Key/Value，避免重复计算 |
| Memory-bound | Decode 阶段的瓶颈是显存带宽，不是算力 |
| PagedAttention | 借鉴 OS 分页，KV Cache 按需分配，消除碎片 |
| Continuous Batching | 请求完成立即替换，GPU 始终满载 |
| Prefix Caching | 共享相同前缀的 KV Cache，减少重复计算 |

### KV Cache 计算公式（必背）

```
KV Cache = 2 × num_layers × num_kv_heads × head_dim × seq_len × dtype_bytes
```

**明天学什么：** Ollama + 本地模型部署——用 GGUF 量化格式在本地跑 LLM。

**写在简历上的要点：**
"熟悉 vLLM 推理加速原理，包括 PagedAttention（借鉴 OS 虚拟内存分页机制优化 KV Cache 显存管理）、Continuous Batching（Iteration-level Scheduling 提升并发吞吐），能独立完成 KV Cache 空间计算和推理瓶颈分析。"